In [1]:
from rapidsegment import StrategicSegmentBuilder, StrategicSegmentScore, UniversalDataLoader

# Example Feature Groupings Setup
variable_groups = {
    "Demographics": [
        "age", 
        "gender", 
        "education_level", 
        "dependents", 
        "home_ownership", 
        "city_tier"
    ],
    "Employment": [
        "employment_type", 
        "years_employed"
    ],
    "Financial_Profile": [
        "annual_income", 
        "credit_score", 
        "existing_debt", 
        "account_balance", 
        "monthly_spend", 
        "loan_to_income_ratio", 
        "previous_defaults", 
        "number_of_cards"
    ],
    "Application_Details": [
        "requested_limit", 
        "application_channel"
    ]
}

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000,5000, 1000, 500, 100],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="is_approved",
    # n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.1,
    min_events = 5,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=variable_groups,
    ignore_features=['application_id'],
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/RapidSegment/credit_app_data.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-17 16:44:16,959 | INFO     | [data_loader.py:147] | 📂 Loading file: /workspaces/RapidSegment/credit_app_data.parquet (extension: .parquet)
2026-07-17 16:44:17,097 | INFO     | [builder.py:403] | 🚀 Starting hierarchical segment extraction...
2026-07-17 16:44:17,114 | INFO     | [builder.py:421] | ⚙️ DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-17 16:44:18,190 | INFO     | [builder.py:153] | ✅ Feature group validation passed. (18 features mapped)
2026-07-17 16:44:18,191 | INFO     | [builder.py:448] | 📊 Dynamic Grid Search Enabled: 10 configurations.
2026-07-17 16:44:18,195 | INFO     | [builder.py:469] | 🔄 Iteration 1 | Remaining Volume: 500,000 | Base Rate: 26.72%
2026-07-17 16:44:18,197 | INFO     | [builder.py:203] | 🔍 Computing IV and bins for 18 features...
2026-07-17 16:44:27,587 | INFO     | [builder.py:661] | 📌 Feature Usage Tracker Update -> 'loan_to_income_ratio' used count = 1
2026-07-17 16:44:27,588 | INFO     | [builder.py:684] | ✅ Segment 1 Captured (Siz

In [2]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(segments_df).columns)
for _, row in pd.DataFrame(segments_df).iterrows():
    table.add_row(list(row))
print(table)

+------------+-----------------------------------+-----------------------------+-------+-------------------+--------------------+--------------------------+-----------------------+
| segment_id |            rule_string            |          sql_filter         | count |        rate       |        lift        | meta_applied_sample_size | meta_applied_min_lift |
+------------+-----------------------------------+-----------------------------+-------+-------------------+--------------------+--------------------------+-----------------------+
|     1      | loan_to_income_ratio=(-inf, 0.07) | loan_to_income_ratio < 0.07 | 43878 | 69.64538037285199 | 2.6060596448508475 |          10000           |          2.0          |
|     2      |     credit_score=[806.50, inf)    |    credit_score >= 806.50   | 35515 | 54.90919329860622 | 2.430095287906122  |          10000           |          2.0          |
+------------+-----------------------------------+-----------------------------+-------+-------

In [3]:

table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate   |        lift        |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
|   0.0   |   420607.0  |    83562.0    | 19.86700173796442  |      26.7244       | 84.12140000000001 | 0.7434030974676482 |
|   1.0   |   43878.0   |    30559.0    |  69.645380372852   |      26.7244       |       8.7756      | 2.606059644850848  |
|   2.0   |   35515.0   |    19501.0    | 54.909193298606226 |      26.7244       |       7.103       | 2.0546464391569588 |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+


In [4]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   loan_to_income_ratio=(-inf, 0.07)
SQL Filter: loan_to_income_ratio < 0.07
--------------------------------------------------
Segment ID: 2
Raw Rule:   credit_score=[806.50, inf)
SQL Filter: credit_score >= 806.50
--------------------------------------------------


In [5]:
builder.explain_feature_journey("city_tier")

📌 AUDIT TRAIL FOR FEATURE: 'city_tier'

[Iteration 1]
  • Current dynamic IV   : 0.0012
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : loan_to_income_ratio=(-inf, 0.07) (Variables: ['loan_to_income_ratio'])

[Iteration 2]
  • Current dynamic IV   : 0.0028
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : credit_score=[806.50, inf) (Variables: ['credit_score'])

[Iteration 3]
  • Current dynamic IV   : 0.0017
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search


In [6]:
# Preparing the dataset for scoring and decile banding.
import duckdb
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN loan_to_income_ratio < 0.08
            THEN 1 ELSE 0 END AS seg_1,
            CASE WHEN credit_score >= 821.50
            THEN 1 ELSE 0 END AS seg_2
            FROM predicted
""").df()
conn.close()

In [7]:
scorer = StrategicSegmentScore(
    target_col="is_approved",
    primary_key="application_id",
    segment_cols=["seg_1","seg_2"],
)

In [8]:
model_artifact = scorer.calculate_and_export_weights(predicted)

2026-07-17 16:44:48,418 | INFO     | [scorer.py:67] | 🚀 Initialising out‑of‑core DuckDB scorecard engine...
2026-07-17 16:44:50,163 | INFO     | [scorer.py:109] | 📊 Computing harmonic scorecard weights...
2026-07-17 16:44:50,164 | INFO     | [scorer.py:147] | ⚡ Scoring population natively via SQL engine...
2026-07-17 16:44:50,451 | INFO     | [scorer.py:164] | 📉 Dataset Zero‑Inflation Rate: 73.28%
2026-07-17 16:44:50,452 | INFO     | [scorer.py:169] | 📈 Calibrating deciles across active populations...
2026-07-17 16:44:50,507 | INFO     | [scorer.py:217] | ✅ Scorecard exported to: scored_experiment_2026_07_17_16_44_16.json


In [9]:
model_artifact.get("decile_min_thresholds")

{'1': 139,
 '2': 97,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [10]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 97
Segment: seg_2 | Weight: 42


In [11]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 98 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 42 ELSE 0 END AS seg_2_weighted
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=140 THEN 1
                    WHEN total_weight >= 98 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [12]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(is_approved) AS events, 
                    (SUM(is_approved)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,1,2541,2517.0,99.055490
1,2,47491,31983.0,67.345392
2,3,449968,99122.0,22.028678
